# NTL passage generation pilot — Model Proxy

50 prompts of our own (the six topics the host uses), three models (`gpt-5.6-luna`, `-sol`, `-terra`),
one passage each. The point is not the data yet — it is to **measure the real cost per passage** and
to check that what comes back looks like a host passage (single paragraph, 40-250 words, no markdown).

No competition text is used or produced here.

In [ ]:
"""Generation pilot: host-format passages for the GPT-5.6 trio through Kaggle's Model Proxy.

Why these three: in our own out-of-fold numbers gpt-5.6-sol sits at 0.17 recall, luna at 0.25 and
terra at 0.26, and every public member makes the same mistake on them. They are also the only weak
classes that the Model Proxy actually serves (glm-5.3, glm-5.3-flash, claude-fable-5, kimi-k3,
gemini-3.7-flash and qwen3.8-max are not in the catalogue).

Fifty prompts of our own, spread over the six topics the host uses. No competition text is involved.
Every call's cost is read back from the SDK usage object, so the next run is sized on measurement.
"""
import json, os, subprocess, sys, time

# The Kaggle image ships an old CLI (2.0.2) without the `benchmarks` group, and the proxy
# credentials only come from that command. Upgrade first, then ask for the token.
print(subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                      "kaggle", "kaggle-benchmarks", "pandas", "pyarrow"],
                     capture_output=True, text=True).stderr[-400:], flush=True)
_auth = subprocess.run([sys.executable, "-m", "kaggle", "benchmarks", "auth", "-y", "--env-file", ".env"],
                       capture_output=True, text=True)
print(f"auth rc={_auth.returncode}\nstdout: {_auth.stdout[-600:]}\nstderr: {_auth.stderr[-600:]}", flush=True)
if not os.path.exists(".env"):
    raise SystemExit("kaggle benchmarks auth produced no .env file — see stderr above")
for line in open(".env"):
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        k, v = line.split("=", 1)
        os.environ[k.strip()] = v.strip().strip('"').strip("'")
print("proxy url set:", bool(os.environ.get("MODEL_PROXY_URL")), flush=True)

import pandas as pd
import kaggle_benchmarks as kbench

MODELS = ["gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra"]
N_PROMPTS = int(os.environ.get("N_PROMPTS", "50"))
OUT = "pilot_passages.parquet"

PROMPTS = [
    # philosophy
    "Explain the difference between knowing a fact and understanding an idea, in a single paragraph.",
    "Is it possible to be mistaken about your own motives? Argue one side in a short paragraph.",
    "What does it mean for a life to go well? Answer in one paragraph without listing.",
    "Describe the relationship between freedom and responsibility in a short passage.",
    "Can a society be unjust even if every law is followed? Answer briefly.",
    "What is the difference between pleasure and happiness? One paragraph.",
    "Argue that doubt is a kind of commitment, in a short paragraph.",
    "Explain what makes a promise binding, in a paragraph.",
    # physics
    "Explain why the sky is blue without using the word scattering, in a short paragraph.",
    "What would change about everyday life if the speed of light were a thousand times slower?",
    "Describe what entropy means to someone who has never studied physics.",
    "Explain why a spinning top stays upright, in one paragraph.",
    "What is the difference between mass and weight, explained in a short paragraph?",
    "Describe how a refrigerator moves heat, in a paragraph.",
    "Explain why the night sky is dark, in a short passage.",
    # mathematics
    "Explain why there are infinitely many prime numbers, in a short paragraph.",
    "What is a proof, and why does mathematics need them? One paragraph.",
    "Describe an intuitive reason the square root of two is not a fraction.",
    "Explain the difference between a conjecture and a theorem, briefly.",
    "What does it mean for a function to be continuous? Answer in a paragraph.",
    "Describe why zero was a difficult idea historically, in a short passage.",
    # biology
    "Explain how a cell knows when to divide, in a short paragraph.",
    "Why do organisms age? Answer in one paragraph.",
    "Describe the difference between a virus and a bacterium for a general reader.",
    "Explain how vaccines train the immune system, in a paragraph.",
    "What is the role of chance in evolution? Answer briefly.",
    "Describe how plants decide when to flower, in a short paragraph.",
    # computer science
    "Explain what makes a sorting algorithm fast, in a short paragraph.",
    "Describe the difference between a process and a thread to a beginner.",
    "What is a hash collision and why does it matter? One paragraph.",
    "Explain why floating point arithmetic produces surprising results.",
    "Describe how a database keeps data consistent during a crash, in a paragraph.",
    "What does it mean for a program to be correct? Answer briefly.",
    "Explain the idea behind public key cryptography in a paragraph.",
    # creative prose
    "Write a short paragraph about a lighthouse keeper who has stopped counting the days.",
    "Describe a kitchen at four in the morning in one paragraph.",
    "Write a paragraph about a train that arrives at a station nobody remembers building.",
    "Describe the sound of a city waking up, in one paragraph.",
    "Write a short paragraph about a library that rearranges itself at night.",
    "Describe a person learning to swim as an adult, in one paragraph.",
    "Write a paragraph about the last shop on a street that is being demolished.",
    "Describe an argument between two people who agree with each other, in one paragraph.",
    "Write a short paragraph about a map that is more accurate than the land.",
    "Describe a garden that grows only at night, in one paragraph.",
    # more mixed, to reach fifty
    "Explain why some problems are harder than others, in a short paragraph.",
    "Describe what a proof by contradiction is, briefly.",
    "Explain the greenhouse effect without naming any gas, in a paragraph.",
    "What is the difference between weather and climate? One paragraph.",
    "Describe how memory works at a high level, in a short paragraph.",
    "Explain why some medicines must be taken at the same time every day, in a paragraph.",
]
PROMPTS = PROMPTS[:N_PROMPTS]
print(f"{len(PROMPTS)} prompts, {len(MODELS)} models = {len(PROMPTS) * len(MODELS)} calls", flush=True)
print("available models (first 10):", list(kbench.llms)[:10], flush=True)

## Generate and measure

In [ ]:
rows, t0 = [], time.time()
for slug in MODELS:
    if slug not in kbench.llms:
        print(f"{slug}: NOT in the proxy catalogue, skipping", flush=True)
        continue
    chat = kbench.llms[slug]
    for i, p in enumerate(PROMPTS):
        try:
            text = chat.prompt(p)
            u = getattr(chat, "usage", None)
            rows.append({"model": slug, "prompt_id": i, "prompt": p, "text": str(text).strip(),
                         "cost_nanodollars": int(getattr(u, "total_cost_nanodollars", 0) or 0)
                         if u is not None else 0})
        except Exception as e:
            rows.append({"model": slug, "prompt_id": i, "prompt": p, "text": "", "error": f"{type(e).__name__}: {e}"})
        if (i + 1) % 10 == 0:
            print(f"  {slug}: {i + 1}/{len(PROMPTS)} done, {time.time() - t0:.0f}s", flush=True)

df = pd.DataFrame(rows)
df.to_parquet(OUT, index=False)
ok = df[df.text.str.len() > 0]
wc = ok.text.str.split().str.len()
summary = {
    "calls": int(len(df)),
    "usable": int(len(ok)),
    "per_model": ok.model.value_counts().to_dict(),
    "median_words": float(wc.median()) if len(wc) else 0,
    "words_in_40_250": int(wc.between(40, 250).sum()) if len(wc) else 0,
    "cost_nanodollars_total": int(df.get("cost_nanodollars", pd.Series([0])).sum()),
    "cost_usd_total": round(int(df.get("cost_nanodollars", pd.Series([0])).sum()) / 1e9, 4),
    "minutes": round((time.time() - t0) / 60, 1),
}
if len(ok):
    summary["cost_usd_per_passage"] = round(summary["cost_usd_total"] / max(len(ok), 1), 6)
    summary["cost_usd_per_passage_est"] = round(summary["cost_usd_total"] / max(len(ok), 1), 6)
json.dump(summary, open("pilot_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2), flush=True)
print("\n--- three samples ---", flush=True)
for _, r in ok.head(3).iterrows():
    print(f"[{r.model}] {r.text[:300]}", flush=True)